# ExTetris Studio — No-Code Game Variant Designer

This notebook lets a game designer explore all 110 generated 4-polyplets, adjust game parameters visually (without writing Python game logic), validate rules via `extetris.config`, preview piece distributions, export a standalone variant bundle with documentation, and run a simulation.

> **Architecture Note:** All validation, piece generation, distributions, and gameplay execute directly through the `extetris` package (`extetris.config` and `extetris.ExTetrisBuilder`)—zero game rules are duplicated in this notebook.

In [1]:
from collections import Counter
from pathlib import Path
import subprocess
import sys
import tempfile

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from extetris import (
    ConfigValidationError,
    ExTetrisBuilder,
    ExTetrominoFactory,
    load_config,
    validate_config,
)

print("Loaded extetris package from:", REPO_ROOT)

Loaded extetris package from: /Users/vanshsharma/Documents/POPL


## 1. Inspect the 110 Generated 4-Polyplets
Explore bounding boxes (including `3x4` and `4x3`), morphology (`OrthogonalTetromino` vs. `DiagonalPolyplet`), and landing-cavity classes.

In [2]:
factory = ExTetrominoFactory()
print("Catalog Summary:", factory.summary())

all_pieces = factory.all_fixed()
bbox_counts = Counter(f"{p.height}x{p.width}" for p in all_pieces)
hole_counts = Counter(p.landing_holes() for p in all_pieces)
print("Bounding Boxes (110 fixed):", dict(sorted(bbox_counts.items())))
print("Landing Holes (110 fixed):", dict(sorted(hole_counts.items())))

print("\nSample 3x4 Diagonal Polyplet (ID 78):")
p78 = factory.by_id(78)
print(f"Class: {p78.__class__.__name__}, holes={p78.landing_holes()}, profile={p78.landing_hole_profile()}")
print(p78)

Catalog Summary: {'fixed_total': 110, 'one_sided_total': 34, 'free_total': 22, 'classic_one_sided': 7, 'classic_fixed': 19, 'diagonal_one_sided': 27, 'diagonal_fixed': 91, 'generation_time_sec': 0.02245}
Bounding Boxes (110 fixed): {'1x4': 1, '2x2': 1, '2x3': 12, '2x4': 14, '3x2': 12, '3x3': 33, '3x4': 10, '4x1': 1, '4x2': 14, '4x3': 10, '4x4': 2}
Landing Holes (110 fixed): {0: 8, 1: 21, 2: 29, 3: 27, 4: 17, 5: 6, 6: 2}

Sample 3x4 Diagonal Polyplet (ID 78):
Class: Box3x4Shape_DiagonalPolyplet_MultiCavityShape, holes=3, profile=(3, 5, 5, 3)
□       
  □     
    □ □ 



## 2. Interactive No-Code Variant Configurator
Load an existing JSON preset (`configs/extended.json`) and customize parameters via `ipywidgets` (with clean non-interactive defaults when executed headlessly).

In [3]:
preset_cfg = load_config(REPO_ROOT / "configs" / "extended.json")

studio_config = {
    "name": "Studio Custom Diagonal Variant",
    "description": "Created in ExTetris Studio without editing Python source.",
    "variant": "custom",
    "piece_scope": "one_sided",
    "connectivity": "diagonal",
    "max_holes": 3,
    "board": {"width": 12, "height": 20},
    "gameplay": {
        "initial_speed_ms": 800,
        "speed_factor": 0.92,
        "enable_wall_kicks": True,
        "enable_ghost": True,
    },
    "distribution": {
        "strategy": "adaptive",
        "bag_multiplier": 1,
        "repeat_penalty": 0.1,
        "seed": 42,
    },
    "renderer": "headless",
}
studio_config = validate_config(studio_config)

try:
    import ipywidgets as widgets
    from IPython.display import display

    w_variant = widgets.Dropdown(options=["classic", "extended", "chaos", "custom"], value=studio_config["variant"], description="Variant:")
    w_scope = widgets.Dropdown(options=["one_sided", "fixed"], value=studio_config["piece_scope"], description="Scope:")
    w_conn = widgets.Dropdown(options=["any", "orthogonal", "diagonal"], value=studio_config["connectivity"], description="Connect:")
    w_holes = widgets.IntSlider(value=studio_config["max_holes"] or 6, min=0, max=6, description="Max Holes:")
    w_width = widgets.IntSlider(value=studio_config["board"]["width"], min=6, max=20, description="Width:")
    w_height = widgets.IntSlider(value=studio_config["board"]["height"], min=10, max=30, description="Height:")
    w_dist = widgets.Dropdown(options=["uniform", "bag", "adaptive", "markov"], value=studio_config["distribution"]["strategy"], description="Dist:")
    w_speed = widgets.IntSlider(value=studio_config["gameplay"]["initial_speed_ms"], min=200, max=1500, step=50, description="Speed(ms):")
    w_out = widgets.Output()

    def _sync_config(*_args):
        candidate = {
            "name": studio_config["name"],
            "description": studio_config["description"],
            "variant": w_variant.value,
            "piece_scope": w_scope.value,
            "connectivity": w_conn.value,
            "max_holes": w_holes.value,
            "board": {"width": w_width.value, "height": w_height.value},
            "gameplay": {
                "initial_speed_ms": w_speed.value,
                "speed_factor": 0.92,
                "enable_wall_kicks": True,
                "enable_ghost": True,
            },
            "distribution": {"strategy": w_dist.value, "seed": 42},
            "renderer": "headless",
        }
        with w_out:
            w_out.clear_output()
            try:
                validated = validate_config(candidate)
                studio_config.clear()
                studio_config.update(validated)
                pool = ExTetrisBuilder(studio_config).build_piece_pool()
                print(f"Valid config! Active shapes in pool: {len(pool)}")
            except ConfigValidationError as err:
                print(err)

    for w in (w_variant, w_scope, w_conn, w_holes, w_width, w_height, w_dist, w_speed):
        w.observe(_sync_config, names="value")

    display(widgets.VBox([
        widgets.HBox([w_variant, w_scope, w_conn]),
        widgets.HBox([w_holes, w_width, w_height]),
        widgets.HBox([w_dist, w_speed]),
        w_out,
    ]))
except ImportError:
    print("ipywidgets not installed; using studio_config dictionary directly.")

print("Active validated studio_config:", studio_config["name"], "| board=", studio_config["board"], "| strategy=", studio_config["distribution"]["strategy"])

Active validated studio_config: Studio Custom Diagonal Variant | board= {'width': 12, 'height': 20} | strategy= adaptive


## 3. Configuration Validation Demo (Rejecting Invalid Designer Inputs)

In [4]:
for label, bad in [
    ("Too-narrow board", {"board": {"width": 2, "height": 20}}),
    ("Classic + diagonal conflict", {"variant": "classic", "connectivity": "diagonal"}),
]:
    try:
        validate_config(bad)
    except ConfigValidationError as exc:
        print(f"[{label}] ->", str(exc).splitlines()[1].strip())

[Too-narrow board] -> - Invalid board.width=2: must be an integer between 4 and 40.
[Classic + diagonal conflict] -> - Incompatible settings: variant='classic' contains only orthogonal shapes and cannot use connectivity='diagonal'.


## 4. Preview Filtered Pool & Sampled Distribution

In [5]:
builder = ExTetrisBuilder(studio_config)
pool = builder.build_piece_pool()
dist = builder.build_distribution(pool=pool)
samples = dist.sample_many(30)

print(f"Filtered pool size: {len(pool)} shapes (all diagonal={all(p.is_diagonal() for p in pool)}, max_holes<={studio_config['max_holes']})")
print("Sampled 30 pieces -> bounding boxes:", dict(Counter(f"{p.height}x{p.width}" for p in samples)))
print("Sampled 30 pieces -> landing holes:", dict(sorted(Counter(p.landing_holes() for p in samples).items())))

Filtered pool size: 16 shapes (all diagonal=True, max_holes<=3)
Sampled 30 pieces -> bounding boxes: {'3x3': 10, '2x3': 4, '2x4': 16}
Sampled 30 pieces -> landing holes: {1: 4, 2: 10, 3: 16}


## 5. Export Variant Bundle & Run Headless Simulation

In [6]:
with tempfile.TemporaryDirectory() as tmp_dir:
    export_dir = Path(tmp_dir) / "studio_release"
    result = builder.export_variant(export_dir)
    print("Exported files:", {k: str(v.name) if hasattr(v, "name") else v for k, v in result.items()})
    proc = subprocess.run([sys.executable, str(result["launcher"])], capture_output=True, text=True, check=True)
    print("Standalone launcher output tail:")
    print("\n".join(proc.stdout.strip().splitlines()[-6:]))

engine, renderer = builder.with_renderer("headless").build()
frames = renderer.run(engine, steps=12, action_sequence=["LEFT", "CW", "HARD_DROP", "RIGHT", "STEP"])
print("\nStudio Headless Simulation Final Frame:")
print(frames[-1])

Exported files: {'config': 'variant_config.json', 'docs': 'VARIANT_README.md', 'launcher': 'run_variant.py', 'piece_count': 16}


Standalone launcher output tail:
|. . . . : . : . . . . . |
|. . . . : : . . . . . . |
|. . . . # # . . . . . . |
|. . . . . # . . . . . . |
|. . . . . . # . . . . . |
+------------------------+

Studio Headless Simulation Final Frame:
State=RUNNING Score=34 Lines=0 Level=1
+------------------------+
|. . . . . . . . . . . . |
|. . . . . . . . . . . . |
|. . . . . . . . . . . . |
|. . . . . . . . . . . . |
|. . . . . . . . . . . . |
|. . . . . . . . . . . . |
|. . . . . . . . . . . . |
|. . . . . . . . . . . . |
|. . . . . @ . @ . . . . |
|. . . . . @ @ . . . . . |
|. . . . . . . . . . . . |
|. . . . . . . . . . . . |
|. . . . . . . . . . . . |
|. . . . . . . . . . . . |
|. . . . . . . . . . . . |
|. . . . . : . : . . . . |
|. . . . . : : . . . . . |
|. . . . . # . . . . . . |
|. . . . # # . . . . . . |
|. . . # . . . . . . . . |
+------------------------+
